In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
nltk.download("punkt")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from gensim.models import Word2Vec, KeyedVectors

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ashwi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
df = pd.read_csv("D:\\news\\Classfication\\news.tsv", sep="\t")
print(df.head())

  News ID Category         Topic  \
0  N10000   sports        soccer   
1  N10001     news  newspolitics   
2  N10002     news        newsus   
3  N10003     news  newspolitics   
4  N10004     news     newsworld   

                                            Headline  \
0  Predicting Atlanta United's lineup against Col...   
1  Mitch McConnell: DC statehood push is 'full bo...   
2            Home In North Highlands Damaged By Fire   
3  Meghan McCain blames 'liberal media' and 'thir...   
4                            Today in History: Aug 1   

                                           News body  \
0  Only FIVE internationals allowed, count em, FI...   
1  WASHINGTON -- Senate Majority Leader Mitch McC...   
2  NORTH HIGHLANDS (CBS13)   Fire damaged a home ...   
3  Meghan McCain is speaking out after a journali...   
4  1714: George I becomes King Georg Ludwig, Elec...   

                                Title entity  \
0  {"Atlanta United's": 'Atlanta United FC'}   
1            

In [3]:
df.isnull().sum()

News ID            0
Category           0
Topic              0
Headline           0
News body         58
Title entity       0
Entity content     0
dtype: int64

In [4]:
df = df.dropna(subset=["Headline", "News body"]).reset_index(drop=True)

In [5]:
df.isnull().sum()

News ID           0
Category          0
Topic             0
Headline          0
News body         0
Title entity      0
Entity content    0
dtype: int64

In [6]:
# Combine Headline + News body
df['text'] = (df['Headline'].str.strip() + ' ' + df['News body'].str.strip()).str.strip()
df['text'] = df['text'].fillna('')

# Feature and target
X = df["text"]
y = df["Category"]

In [7]:
df = df[["Headline", "Title entity", "Entity content"]]

In [8]:
# Take only 50% of the data to reduce computation
df = df.sample(frac=0.5, random_state=42).reset_index(drop=True)

In [9]:
import ast

def parse_dict(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["title_entity_dict"] = df["Title entity"].apply(parse_dict)

In [10]:
import re

def clean_text_ner(text):
    text = re.sub(r"<.*?>", "", str(text))
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["Headline"].apply(clean_text_ner)

In [11]:
def tokenize(text):
    return text.split()

df["tokens"] = df["clean_text"].apply(tokenize)

In [12]:
import spacy

In [13]:
import spacy
nlp = spacy.blank("en")  # ✅ 100% works - no transformers/spacy_transformers needed
df["Headline"] = df["Headline"].astype(str)
df["Title entity"] = df["Title entity"].astype(str)
print("✅ spaCy pipeline ready - tokenization + processing works!")


✅ spaCy pipeline ready - tokenization + processing works!


In [15]:
import spacy
nlp = spacy.blank("en")
df["Headline"] = df["Headline"].astype(str)
df


,Headline,Title entity,Entity content,title_entity_dict,clean_text,tokens
0,This dog's smile will melt your heart,{},{},{},This dog's smile will melt your heart,"[This, dog's, smile, will, melt, your, heart]"
1,The Most Popular Walmart Item in Every State,{'Walmart': 'Walmart'},"{'Walmart': {'type': 'item', 'id': 'Q18615334'...",{'Walmart': 'Walmart'},The Most Popular Walmart Item in Every State,"[The, Most, Popular, Walmart, Item, in, Every,..."
2,Photos: Look Glenn Close's 'Beanfield' estate ...,"{""Glenn Close's"": 'Glenn Close', 'Bedford': 'B...","{'Glenn Close': {'type': 'item', 'id': 'Q37231...","{'Glenn Close's': 'Glenn Close', 'Bedford': 'B...",Photos: Look Glenn Close's 'Beanfield' estate ...,"[Photos:, Look, Glenn, Close's, 'Beanfield', e..."
3,Hillsborough Sheriff's Office sweep results in...,{'human trafficking': 'Human trafficking'},"{'Human trafficking': {'type': 'item', 'id': '...",{'human trafficking': 'Human trafficking'},Hillsborough Sheriff's Office sweep results in...,"[Hillsborough, Sheriff's, Office, sweep, resul..."
4,Family of missing Connecticut mom blast 'Gone ...,{'Connecticut': 'Connecticut'},"{'Connecticut': {'type': 'item', 'id': 'Q58425...",{'Connecticut': 'Connecticut'},Family of missing Connecticut mom blast 'Gone ...,"[Family, of, missing, Connecticut, mom, blast,..."
...,...,...,...,...,...,...
56847,Home and garden events list for the Milwaukee ...,{'Milwaukee': 'Milwaukee'},"{'Milwaukee': {'type': 'item', 'id': 'Q6861641...",{'Milwaukee': 'Milwaukee'},Home and garden events list for the Milwaukee ...,"[Home, and, garden, events, list, for, the, Mi..."
56848,2019 Nissan Maxima: Refreshed for More V-Motion,{'Nissan': 'Nissan'},"{'Nissan': {'type': 'item', 'id': 'Q958359', '...",{'Nissan': 'Nissan'},2019 Nissan Maxima: Refreshed for More V-Motion,"[2019, Nissan, Maxima:, Refreshed, for, More, ..."
56849,US officials want 'El Chapo' to forfeit $12.7 ...,"{'cocaine': 'Cocaine', 'heroin': 'Heroin'}","{'Cocaine': {'type': 'item', 'id': 'Q95581366'...","{'cocaine': 'Cocaine', 'heroin': 'Heroin'}",US officials want 'El Chapo' to forfeit $12.7 ...,"[US, officials, want, 'El, Chapo', to, forfeit..."
56850,All of SC is under a severe thunderstorm watch...,{'SC': 'South Carolina'},"{'South Carolina': {'type': 'item', 'id': 'Q10...",{'SC': 'South Carolina'},All of SC is under a severe thunderstorm watch...,"[All, of, SC, is, under, a, severe, thundersto..."


In [24]:
!python -m spacy download en_core_web_sm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.9.4 which is incompatible.


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 3.0 MB/s eta 0:00:04
     ----- ---------------------------------- 1.8/12.8 MB 3.1 MB/s eta 0:00:04
     --------- ------------------------------ 2.9/12.8 MB 3.8 MB/s eta 0:00:03
     ------------ --------------------------- 3.9/12.8 MB 4.1 MB/s eta 0:00:03
     ----------------- ---------------------- 5.5/12.8 MB 4.6 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/12.8 MB 5.3 MB/s eta 0:00:02
     ------------------------------- -------- 10.0/12.8 MB 6.3 MB/s eta 0:00:01
     ---------------------------------- ----- 11.0/12.8 MB 6.1 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 6.5 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 6.5 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 6.5

In [14]:
nlp = spacy.load("en_core_web_sm")

df["Headline"] = df["Headline"].astype(str)
df["Title entity"] = df["Title entity"].astype(str)

COUNTRIES = ["United States", "India", "Brazil", "China", "Mexico", "Canada"]
PERSON_PATTERN = r"^[A-Z][a-z]+(\s[A-Z][a-z]+)+$"
ORG_KEYWORDS = ["Corporation", "Authority", "Committee", "Association", "University", "Agency", "Company", "FC", "Ltd"]


def infer_entity_type(expanded):
    expanded = expanded.strip()

    if re.match(PERSON_PATTERN, expanded):
        return "PERSON"

    if expanded in COUNTRIES:
        return "LOCATION"

    if any(k in expanded for k in ORG_KEYWORDS):
        return "ORG"

    return "MISC"

def convert_to_bio(text, entity_string):
    tokens = text.split()
    tags = ["O"] * len(tokens)

    if entity_string == "{}":
        return tokens, tags

    try:
        ent_dict = ast.literal_eval(entity_string)
    except:
        return tokens, tags

    lower_tokens = [w.lower().strip(".,!?") for w in tokens]

    
    for surface, expanded in ent_dict.items():
        clean_surface = surface.replace("'s", "").strip()
        stoks = clean_surface.split()
        stoks = [w.lower().strip(".,!?") for w in stoks]
        n = len(stoks)

        ent_type = infer_entity_type(expanded)

        # Search entity span safely
        for i in range(len(tokens)):
            try:
                if lower_tokens[i:i+n] == stoks:
                    tags[i] = f"B-{ent_type}"
                    for j in range(i+1, i+n):
                        if j < len(tags):   # SAFETY CHECK
                            tags[j] = f"I-{ent_type}"
            except:
                continue

    return tokens, tags



sentences = []
labels = []

for _, row in df.iterrows():
    s, t = convert_to_bio(row["Headline"], row["Title entity"])
    sentences.append(s)
    labels.append(t)

print("DATA READY — Samples:", len(sentences))

DATA READY — Samples: 56852


In [16]:
def create_ner_tags(tokens, entity_content, title_entity):

    tags = ["O"] * len(tokens)   # O means Outside entity
    entities = []

    # Add entity_content words
    if isinstance(entity_content, str):
        entities.extend(entity_content.split())

    # Add title_entity words
    if isinstance(title_entity, str):
        entities.extend(title_entity.split())

    # Assign tags
    for i, token in enumerate(tokens):
        if token in entities:
            tags[i] = "B-ENT"

    return tags

In [24]:
print(df.columns)

Index(['Headline', 'Title entity', 'Entity content', 'title_entity_dict',
       'clean_text', 'tokens'],
      dtype='object')


In [25]:
df["tokens"] = df["Headline"].astype(str).apply(lambda x: x.split())

In [26]:
df["ner_tags"] = df.apply(
    lambda row: create_ner_tags(
        row["tokens"],
        row["Entity content"],
        row["Title entity"]
    ),
    axis=1
)

In [19]:
df = df[df["tokens"].apply(len) == df["ner_tags"].apply(len)]

In [27]:
print(df[["tokens","ner_tags"]].head())

                                              tokens  \
0      [This, dog's, smile, will, melt, your, heart]   
1  [The, Most, Popular, Walmart, Item, in, Every,...   
2  [Photos:, Look, Glenn, Close's, 'Beanfield', e...   
3  [Hillsborough, Sheriff's, Office, sweep, resul...   
4  [Family, of, missing, Connecticut, mom, blast,...   

                                         ner_tags  
0                           [O, O, O, O, O, O, O]  
1                        [O, O, O, O, O, O, O, O]  
2                [O, O, B-ENT, O, O, O, B-ENT, O]  
3  [O, O, O, O, O, B-ENT, O, O, O, O, O, O, O, O]  
4                 [O, B-ENT, O, O, O, O, O, O, O]  


In [28]:
from datasets import Dataset

ner_dataset = Dataset.from_pandas(df[["tokens", "ner_tags"]])
ner_dataset

c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 56852
})

In [31]:

# Example BIO labels – adjust if needed
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [35]:
print(set(tag for row in df_small["ner_tags"] for tag in row))

{'B-ENT', 'O'}


In [32]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [33]:
max_len = 128

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],              # list of word tokens
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_len
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)    # padding / special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[labels[word_idx]])
            else:
                label_ids.append(-100)    # subword → ignore

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [36]:
label_list = ["O", "B-ENT"]

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)
print(id2label)

{'O': 0, 'B-ENT': 1}
{0: 'O', 1: 'B-ENT'}


In [37]:

from datasets import Dataset

# Example: you already created these
# tokens → list[list[str]]
# ner_tags → list[list[str]]

df_small = df.sample(frac=0.5, random_state=42)

dataset = Dataset.from_dict({
    "tokens": df_small["tokens"].tolist(),
    "ner_tags": df_small["ner_tags"].tolist()
})

dataset = dataset.train_test_split(test_size=0.2)

tokenized_ds = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map: 100%|██████████| 5686/5686 [00:01<00:00, 3912.99 examples/s]


In [38]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ashwi\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:01<00:00, 150.43it/s, Materializing param=bert.encode

In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_ner",
    max_steps=850,          # 🔴 STOP at step 850
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

In [40]:
import numpy as np
from sklearn.metrics import classification_report

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                true_preds.append(id2label[p_i])
                true_labels.append(id2label[l_i])

    report = classification_report(true_labels, true_preds, output_dict=True)
    return {
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"]
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

c:\Users\ashwi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.229941
100,0.151446


In [ ]:
model.save_pretrained("./bert_ner_final")
tokenizer.save_pretrained("./bert_ner_final")

In [ ]:
import numpy as np
from seqeval.metrics import f1_score